在法国，可以用于构建2015到2025年新闻语料库的网站主要有以下几类：

媒体网站（部分可以自由抓取）

Liberation（liberation.fr），部分文章开放，可以抓取。

20 Minutes（20minutes.fr），免费内容，反爬不强，比较适合。

L’Humanité（humanite.fr），部分文章免费。

Ouest-France（ouest-france.fr），大量新闻，地区性为主，有轻微反爬。

La Croix（la-croix.com），部分新闻开放。

无或反爬较弱的国际媒体

France24（france24.com/fr），档案完整，较易抓取。

RFI（rfi.fr/fr），新闻内容多，覆盖全面。

TV5MONDE（information.tv5monde.com），国际与法国新闻并存。

开源新闻数据项目

GDELT Project（gdeltproject.org），全球新闻数据库，包含大量法语新闻，2015–2025可覆盖。

News on the Web (NOW) Corpus，包含法语新闻，可以按年份导出。

学术和存档资源

Europresse 或 Factiva，需要机构订阅，但可以批量下载新闻。

Internet Archive (Wayback Machine)，可以抓取媒体网站在2015–2025年的历史快照，绕过部分反爬。

不适合直接抓取的媒体：

Le Monde，有付费墙和较强的反爬。

Le Figaro，同样有付费墙和反爬。

Mediapart，完全在付费墙后。

Le Canard enchaîné，几乎没有在线档案，主要是纸质出版，不适合抓取。

结论：如果你要收集大规模的2015–2025法语新闻语料库，推荐首选 France24、RFI、TV5MONDE、20 Minutes 和 GDELT。这些来源覆盖面广且技术上容易抓取。

#1） 20 Minutes（20minutes.fr


# https://www.google.com/advanced_search?q=site:thepaper.cn+2015&client=firefox-b-d&sca_esv=4d1e52ecd0f51357&sxsrf=AE3TifOp3VsAjjic0H6D16Q26gmvMKT8dQ:1753550785198&tbas=0&biw=1232&bih=728&dpr=2



#-  https://www.20minutes.fr/archives/2015

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 目标 URL
url = "https://www.20minutes.fr/politique/4016873-20221231-nouvel-an-emmanuel-macron-appelle-pays-uni-solidaire-v-ux-presidentiels-2023"

# 请求网页
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
response.encoding = "utf-8"  # 防止法语字符乱码

# 解析 HTML
soup = BeautifulSoup(response.text, "html.parser")

# ===== 提取标题 =====
title_tag = soup.find("h1", class_="mb-0@xs")
title = title_tag.get_text(strip=True) if title_tag else ""

# ===== 提取作者 =====
author_tag = soup.find("p", class_="color-black")
author = author_tag.get_text(strip=True) if author_tag else ""

# ===== 提取日期 =====
date_tag = soup.find("time")
date = date_tag.get_text(strip=True) if date_tag else ""

# ===== 提取正文内容 =====
content_div = soup.find("div", class_="c-content")
paragraphs = content_div.find_all("p") if content_div else []
content = "\n".join(p.get_text(strip=True) for p in paragraphs)

# ===== 保存数据到 CSV =====
data = [{
    "author": author,
    "title": title,
    "date": date,
    "content": content,
    "url": url
}]

df = pd.DataFrame(data)
df.to_csv("20minutes2022_24.csv", index=False, encoding="utf-8-sig")

print("已完成抓取，已保存为 20minutes2015_1.csv")


已完成抓取，已保存为 20minutes2015_1.csv


In [ ]:
#下载包


import zipfile
import os

zip_path = "/content/20minutes2023_all_csv.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in os.listdir("/content"):
        if file.endswith(".csv") and file.startswith("20minutes2023_"):
            zipf.write(os.path.join("/content", file), arcname=file)


In [ ]:
from google.colab import files
files.download("/content/20minutes2023_all_csv.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#一次性搜索2015-2025年的所有的20 minutes 网址内容

In [ ]:
#Étape 1 : faire une fonction de scraping pour un article

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urljoin, urlparse
from requests.adapters import HTTPAdapter, Retry

BASE = "https://www.20minutes.fr"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# --- Session robuste (retries) ---
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504])
session.mount("http://", HTTPAdapter(max_retries=retries))
session.mount("https://", HTTPAdapter(max_retries=retries))

In [ ]:
#Étape 2 : récupérer les liens d’une page d’archives

#Sur une page comme https://www.20minutes.fr/archives/2015, tu peux extraire tous les liens d’articles :

# --- 1) Scraper un article ---
def scrape_article(url: str) -> dict:
    resp = session.get(url, headers=HEADERS, timeout=20)
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")

    # Titre (garde un fallback au cas où la classe change)
    title_tag = soup.find("h1") or soup.select_one("h1.mb-0@xs")
    title = title_tag.get_text(strip=True) if title_tag else ""

    # Auteur (fallbacks possibles)
    author = ""
    author_tag = soup.find("p", class_="color-black") or soup.select_one('[rel="author"]')
    if author_tag:
        author = author_tag.get_text(strip=True)
    else:
        meta_author = soup.find("meta", attrs={"name": "author"})
        if meta_author and meta_author.get("content"):
            author = meta_author["content"].strip()

    # Date (time tag ou meta)
    date = ""
    time_tag = soup.find("time")
    if time_tag:
        date = time_tag.get_text(strip=True) or time_tag.get("datetime", "").strip()
    else:
        meta_date = soup.find("meta", attrs={"property": "article:published_time"})
        if meta_date and meta_date.get("content"):
            date = meta_date["content"].strip()

    # Contenu
    content = ""
    content_div = soup.find("div", class_="c-content") or soup.select_one("article")
    if content_div:
        paragraphs = content_div.find_all("p")
        content = "\n".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))

    return {
        "author": author,
        "title": title,
        "date": date,
        "content": content,
        "url": url
    }



In [ ]:
# --- 2) Repérer si un lien ressemble à un article 20minutes ---
ARTICLE_RE = re.compile(r"^https?://www\.20minutes\.fr/(?!archives/)(?!dossier/)(?!video/).+")

def looks_like_article(href: str) -> bool:
    # élimine les ancres et paramètres fantaisie
    if not href or "mailto:" in href or "javascript:" in href:
        return False
    if not ARTICLE_RE.match(href):
        return False
    # filtre quelques rubriques non-articles
    bad_parts = ["/archives/", "/videos/", "/video/", "/services/", "/podcasts/", "/partenaires/"]
    if any(bp in href for bp in bad_parts):
        return False
    return True

def absolutize(base_url: str, href: str) -> str:
    if href.startswith("http"):
        return href
    return urljoin(base_url, href)


In [ ]:
# --- 3) Extraire tous les liens d’articles d’une page d’archives, avec pagination ---
def get_all_article_links_from_archive(archive_url: str, max_pages: int = 0) -> list:
    """
    archive_url : ex. https://www.20minutes.fr/archives/2015
    max_pages   : 0 = pas de limite (parcourt toute la pagination). Mets p.ex. 100 si tu veux une borne.
    """
    seen = set()
    to_visit = [archive_url]
    page_count = 0

    while to_visit:
        url = to_visit.pop(0)
        try:
            resp = session.get(url, headers=HEADERS, timeout=20)
            resp.encoding = "utf-8"
            soup = BeautifulSoup(resp.text, "html.parser")

            # Récupère liens d’articles
            for a in soup.select("a[href]"):
                href = absolutize(url, a["href"])
                if looks_like_article(href):
                    seen.add(href)

            # Trouve un lien "page suivante" (selon structure probable)
            next_link = None
            # 1) rel=next si présent
            nl = soup.select_one('a[rel="next"]')
            if nl and nl.get("href"):
                next_link = absolutize(url, nl["href"])
            else:
                # 2) texte "Suivant" / "Plus" / symbole >
                cand = soup.find("a", string=re.compile(r"(Suivant|Plus|>|\u00BB)", re.IGNORECASE))
                if cand and cand.get("href"):
                    next_link = absolutize(url, cand["href"])

            # Évite boucles + respecte max_pages
            if next_link and next_link not in to_visit and (max_pages == 0 or page_count < max_pages):
                to_visit.append(next_link)
                page_count += 1

            # Politesse
            time.sleep(0.7)

        except Exception as e:
            print("[Archive] Erreur sur", url, "->", e)

    return sorted(seen)

In [ ]:
# ... 这里是所有函数的定义，包括 get_all_article_links_from_archive ...

from tqdm import tqdm
import time
import pandas as pd
from urllib.parse import urlparse

def main():
    year_urls = [
        "https://www.20minutes.fr/archives/2015",

    ]

    t0 = time.time()  # 总开始时间

    for yurl in year_urls:
        year = urlparse(yurl).path.strip("/").split("/")[-1]
        print(f"\n=== Année {year} : collecte des liens depuis {yurl} ===")
        links = get_all_article_links_from_archive(yurl, max_pages=0)

        print(f"→ {len(links)} liens d’articles détectés pour {year}")
        rows = []
        start_year = time.time()

        for link in tqdm(links, desc=f"Scraping {year}", unit="article"):
            try:
                art = scrape_article(link)
                rows.append(art)
                time.sleep(1.0)  # politesse serveur
            except Exception as e:
                print("[Article] Erreur sur", link, "->", e)

        # 保存
        df = pd.DataFrame(rows)
        out = f"20minutes_{year}.csv"
        df.to_csv(out, index=False, encoding="utf-8-sig")
        elapsed_year = time.time() - start_year
        print(f"✔ Sauvegardé : {out} ({len(df)} lignes) en {elapsed_year:.1f} sec")

    total_elapsed = time.time() - t0
    print(f"\n Temps total : {total_elapsed/60:.1f} minutes")

if __name__ == "__main__":
    main()



=== Année 2015 : collecte des liens depuis https://www.20minutes.fr/archives/2015 ===


In [ ]:
#这个不计时


# --- 4) Boucler sur 2015 → 2025 ---
def fix_year_url(u: str) -> str:
    # corrige un éventuel "www.20minutes.fr/archives/2020" sans https
    if u.startswith("www."):
        return "https://" + u
    return u

def main():
    year_urls = [
        "https://www.20minutes.fr/archives/2015",
        "https://www.20minutes.fr/archives/2016",
        "https://www.20minutes.fr/archives/2017",
        "https://www.20minutes.fr/archives/2018",
        "https://www.20minutes.fr/archives/2019",
        "https://www.20minutes.fr/archives/2020",   # corrige le tien: ajout de https://
        "https://www.20minutes.fr/archives/2021",
        "https://www.20minutes.fr/archives/2022",
        "https://www.20minutes.fr/archives/2023",
        "https://www.20minutes.fr/archives/2024",
        "https://www.20minutes.fr/archives/2025",
    ]

    for yurl in year_urls:
        yurl = fix_year_url(yurl)
        year = urlparse(yurl).path.strip("/").split("/")[-1]
        print(f"=== Année {year} : collecte des liens depuis {yurl} ===")

        links = get_all_article_links_from_archive(yurl, max_pages=0)  # 0 = tout parcourir
        print(f"→ {len(links)} liens d’articles détectés pour {year}")

        rows = []
        for i, link in enumerate(links, 1):
            try:
                art = scrape_article(link)
                rows.append(art)
                if i % 20 == 0:
                    print(f"  {i}/{len(links)} articles...")
                time.sleep(1.0)  # politesse serveur
            except Exception as e:
                print("[Article] Erreur sur", link, "->", e)

        df = pd.DataFrame(rows)
        out = f"20minutes_{year}.csv"
        df.to_csv(out, index=False, encoding="utf-8-sig")
        print(f"✔ Sauvegardé : {out} ({len(df)} lignes)")

if __name__ == "__main__":
    main()


In [ ]:
#保存

from google.colab import drive
drive.mount('/content/drive')

# 修改保存路径
out = f"/content/drive/MyDrive/20minutes_{year}.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Sauvegardé dans Google Drive : {out}")


In [ ]:
#另一个代码  试试

import requests
from bs4 import BeautifulSoup
import time
import csv

base_url = "https://www.20minutes.fr/archives/"
years = range(2015, 2026)

results = []

headers = {"User-Agent": "Mozilla/5.0"}

for year in years:
    year_url = f"{base_url}{year}"
    print(f"\n=== 抓取年份: {year} ===")

    try:
        r = requests.get(year_url, headers=headers, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"获取年份页面失败: {year_url} - {e}")
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    # 找所有文章链接
    article_links = [a["href"] for a in soup.select("a[href*='20minutes.fr/']") if "article" in a["href"]]
    article_links = list(set(article_links))  # 去重

    for link in article_links:
        try:
            r_article = requests.get(link, headers=headers, timeout=10)
            r_article.raise_for_status()
        except Exception as e:
            print(f"文章失败: {link} - {e}")
            continue

        soup_article = BeautifulSoup(r_article.text, "html.parser")

        # 抓标题
        title = soup_article.find("h1")
        title = title.get_text(strip=True) if title else ""

        # 抓正文
        paragraphs = soup_article.select("div.article-body p")
        text = "\n".join(p.get_text(strip=True) for p in paragraphs)

        if text.strip():
            results.append([year, link, title, text])
            print(f"  抓取成功: {title[:30]}...")

        time.sleep(0.5)  # 防止封锁

# 保存 CSV
with open("20minutes_2015_2025.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["year", "url", "title", "text"])
    writer.writerows(results)

print("\n 完成！已保存到 20minutes_2015_2025.csv")


#没有反爬虫
#Euronews 法语版
#Euronews 法语版


In [ ]:
! pip install requests

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

url = "https://fr.euronews.com/business/2026/01/22/marches-en-hausse-valeurs-refuges-en-baisse-trump-vante-un-accord-sur-le-groenland"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "fr-FR,fr;q=0.9"
}

r = requests.get(url, headers=headers, timeout=15)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

# -------- 1. title --------
title_tag = soup.select_one("h1.c-article-redesign-title")
title = title_tag.get_text(strip=True) if title_tag else ""

# -------- 2. date --------
date_tag = soup.find("time")
date = date_tag.get_text(strip=True) if date_tag else ""

# -------- 3. author --------
author_tag = soup.find("b")
author = author_tag.get_text(strip=True) if author_tag else "Euronews"

# -------- 4. content --------
content_parts = []

# summary
summary_tag = soup.select_one("h2.c-article-summary")
if summary_tag:
    content_parts.append(summary_tag.get_text(" ", strip=True))

# paragraphs
for p in soup.select("div.c-article-content p"):
    txt = p.get_text(" ", strip=True)
    if txt:
        content_parts.append(txt)

content_text = "\n\n".join(content_parts)

# -------- save CSV --------
row = [author, title, date, content_text, url]

out_file = "euronews_2025_22.csv"
with open(out_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["author", "title", "date", "content", "url"])
    writer.writerow(row)

print(" 完成")
print("Title:", title)
print("Date:", date)
print("Author:", author)
print("Paragraphs:", len(content_parts))
print("Saved to:", out_file)


 完成
Title: Marchés en hausse, valeurs refuges en baisse, Trump vante un accord sur le Groenland
Date: 22/01/2026 - 10:31 UTC+1
Author: Related
Paragraphs: 18
Saved to: euronews_2025_22.csv


#打包下载

In [ ]:
import zipfile
import os

zip_name = "euronews_2025_articles.zip"

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for i in range(1, 23):
        filename = f"euronews_2025_{i}.csv"
        if os.path.exists(filename):
            zipf.write(filename)
        else:
            print("未找到：", filename)

print(" ZIP 文件已生成:", zip_name)


 ZIP 文件已生成: euronews_2025_articles.zip


In [ ]:
from google.colab import files
files.download("euronews_2025_articles.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#3) Le Figaro


In [ ]:
!pip install requests beautifulsoup4


In [195]:
import requests
from bs4 import BeautifulSoup
import csv

url = "https://www.lefigaro.fr/langue-francaise/actu-des-mots/labubu-masculinisme-conclave-ces-mots-qui-ont-marque-l-annee-2025-20251228"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=15)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

# -------- title --------
title_tag = soup.select_one("h1.fig-headline")
title = title_tag.get_text(strip=True) if title_tag else ""

# -------- date --------
date = ""
time_tag = soup.find("time")
if time_tag:
    date = time_tag.get("datetime", "").strip()

# -------- author --------
author = ""
author_tag = soup.select_one("a.fig-content-metas__author")
if author_tag:
    author = author_tag.get_text(strip=True)

# -------- content --------
content_parts = []

# 导语
standfirst = soup.select_one("p.fig-standfirst")
if standfirst:
    content_parts.append(standfirst.get_text(" ", strip=True))

# 正文段落
for p in soup.select("p.fig-paragraph"):
    txt = p.get_text(" ", strip=True)
    if txt:
        content_parts.append(txt)

content_text = "\n\n".join(content_parts)

# -------- source --------
source = "Le Figaro"

# -------- save to CSV --------
output_file = "lefigaro2025_article24.csv"

with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["author", "title", "date", "content", "source", "url"])
    writer.writerow([author, title, date, content_text, source, url])

# -------- print result --------
print("抓取成功")
print("Title:", title)
print("Author:", author)
print("Date:", date)
print("Content length:", len(content_text))
print("Saved as:", output_file)


抓取成功
Title: « Labubu », « masculinisme », « conclave »… Ces mots qui ont marqué l’année 2025
Author: Alice Develey
Date: 2025-12-28T10:39:38+01:00
Content length: 1051
Saved as: lefigaro2025_article24.csv


In [201]:
import zipfile
import os

zip_name = "lefigaro2025_articles.zip"

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for i in range(1, 21):
        filename = f"lefigaro2025_article{i}.csv"
        if os.path.exists(filename):
            zipf.write(filename)
            print("已加入:", filename)
        else:
            print("未找到:", filename)

print("ZIP 文件已生成:", zip_name)


已加入: lefigaro2025_article1.csv
已加入: lefigaro2025_article2.csv
已加入: lefigaro2025_article3.csv
已加入: lefigaro2025_article4.csv
已加入: lefigaro2025_article5.csv
已加入: lefigaro2025_article6.csv
已加入: lefigaro2025_article7.csv
已加入: lefigaro2025_article8.csv
已加入: lefigaro2025_article9.csv
已加入: lefigaro2025_article10.csv
已加入: lefigaro2025_article11.csv
已加入: lefigaro2025_article12.csv
已加入: lefigaro2025_article13.csv
已加入: lefigaro2025_article14.csv
已加入: lefigaro2025_article15.csv
已加入: lefigaro2025_article16.csv
已加入: lefigaro2025_article17.csv
已加入: lefigaro2025_article18.csv
已加入: lefigaro2025_article19.csv
已加入: lefigaro2025_article20.csv
ZIP 文件已生成: lefigaro2025_articles.zip


In [202]:
from google.colab import files
files.download("lefigaro2025_articles.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#2） France24（france24.com/fr

In [ ]:
! pip install requests beautifulsoup4



 抓取日期页: https://www.france24.com/fr/archives/2015/01/30-janvier-2015
发现文章数: 0

 完成，保存为: france24_archive2015_1.csv


#函数 2：抓取单篇文章内容

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

url = "https://www.france24.com/fr/archives/2015/01/30-janvier-2015"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers, timeout=15)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

# 1. titre (archive page title)
title_tag = soup.find("span")
title = title_tag.get_text(strip=True) if title_tag else ""

# 2. date
date_tag = soup.select_one("a.o-archive-day__nav__link")
date = date_tag.get_text(strip=True) if date_tag else ""

# 3. content (list of article titles)
contents = []
for li in soup.select("ul.o-archive-day__list li.o-archive-day__list__entry"):
    h2 = li.find("h2")
    if h2:
        contents.append(h2.get_text(strip=True))

content_text = " | ".join(contents)

# 4. author (empty)
author = ""

row = [author, title, date, content_text, url]

# Save to CSV
out_file = "france24_2015_1.csv"
with open(out_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["author", "title", "date", "content", "url"])
    writer.writerow(row)

print(" 完成")
print("Title:", title)
print("Date:", date)
print("Articles:", len(contents))
print("Saved to:", out_file)


HTTPError: 403 Client Error: Forbidden for url: https://www.france24.com/fr/archives/2015/01/30-janvier-2015

#❌ France24 对普通 requests 访问返回 403 Forbidden（反爬虫）

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

url = "https://www.france24.com/fr/archives/2015/01/30-janvier-2015"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.france24.com/",
    "Connection": "keep-alive",
}

r = requests.get(url, headers=headers, timeout=15)
print("Status code:", r.status_code)  # 调试用

r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

# titre
title_tag = soup.find("span")
title = title_tag.get_text(strip=True) if title_tag else ""

# date
date_tag = soup.select_one("a.o-archive-day__nav__link")
date = date_tag.get_text(strip=True) if date_tag else ""

# content
contents = []
for li in soup.select("ul.o-archive-day__list li.o-archive-day__list__entry"):
    h2 = li.find("h2")
    if h2:
        contents.append(h2.get_text(strip=True))

content_text = " | ".join(contents)

author = ""

row = [author, title, date, content_text, url]

out_file = "france24_2015_1.csv"
with open(out_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["author", "title", "date", "content", "url"])
    writer.writerow(row)

print(" 完成")
print("Title:", title)
print("Date:", date)
print("Articles:", len(contents))
print("Saved to:", out_file)


Status code: 403


HTTPError: 403 Client Error: Forbidden for url: https://www.france24.com/fr/archives/2015/01/30-janvier-2015

#3） RFI（rfi.fr/fr）

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

url = "https://www.rfi.fr/fr/sport/20260118-un-penalty-un-boycott-une-panenka-rat%C3%A9e-les-folles-minutes-de-la-finale-s%C3%A9n%C3%A9gal-maroc?dicbo=v2-oY05iUN"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
    "Referer": "https://www.rfi.fr/",
}

r = requests.get(url, headers=headers, timeout=15)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

# -------- 1. Title --------
title_tag = soup.select_one("h1.a-page-title")
title = title_tag.get_text(strip=True) if title_tag else ""

# -------- 2. Date --------
date_tag = soup.find("time")
date = date_tag.get_text(strip=True) if date_tag else ""

# -------- 3. Author --------
author_tag = soup.select_one("a.m-from-author__name")
author = author_tag.get_text(strip=True) if author_tag else ""

# -------- 4. Content --------
content_parts = []

# chapo
chapo = soup.select_one("p.t-content__chapo")
if chapo:
    content_parts.append(chapo.get_text(" ", strip=True))

# other paragraphs
for p in soup.select("div.t-content p"):
    txt = p.get_text(" ", strip=True)
    if txt and txt not in content_parts:
        content_parts.append(txt)

# strong conclusion
strong_tag = soup.select_one("div.t-content strong")
if strong_tag:
    content_parts.append(strong_tag.get_text(" ", strip=True))

content_text = "\n\n".join(content_parts)

# -------- Save CSV --------
row = [author, title, date, content_text, url]

out_file = "rfi_article_2015_2.csv"
with open(out_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["author", "title", "date", "content", "url"])
    writer.writerow(row)

print(" 完成")
print("Author:", author)
print("Title:", title)
print("Date:", date)
print("Content length:", len(content_text))
print("Saved to:", out_file)


HTTPError: 403 Client Error: Forbidden for url: https://www.rfi.fr/fr/sport/20260118-un-penalty-un-boycott-une-panenka-rat%C3%A9e-les-folles-minutes-de-la-finale-s%C3%A9n%C3%A9gal-maroc?dicbo=v2-oY05iUN

In [ ]:
!pip install cloudscraper


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
import cloudscraper
from bs4 import BeautifulSoup
import csv

url = "https://www.rfi.fr/fr/sport/20260118-un-penalty-un-boycott-une-panenka-rat%C3%A9e-les-folles-minutes-de-la-finale-s%C3%A9n%C3%A9gal-maroc"

scraper = cloudscraper.create_scraper(
    browser={
        "browser": "chrome",
        "platform": "windows",
        "mobile": False
    }
)

r = scraper.get(url, timeout=20)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

# -------- title --------
title_tag = soup.select_one("h1.a-page-title")
title = title_tag.get_text(strip=True) if title_tag else ""

# -------- date --------
date_tag = soup.find("time")
date = date_tag.get_text(strip=True) if date_tag else ""

# -------- author --------
author_tag = soup.select_one("a.m-from-author__name")
author = author_tag.get_text(strip=True) if author_tag else ""

# -------- content --------
content_parts = []

chapo = soup.select_one("p.t-content__chapo")
if chapo:
    content_parts.append(chapo.get_text(" ", strip=True))

for p in soup.select("div.t-content p"):
    txt = p.get_text(" ", strip=True)
    if txt:
        content_parts.append(txt)

strong_tag = soup.select_one("div.t-content strong")
if strong_tag:
    content_parts.append(strong_tag.get_text(" ", strip=True))

content_text = "\n\n".join(content_parts)

# -------- save --------
row = [author, title, date, content_text, url]

out_file = "rfi_article.csv"
with open(out_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["author", "title", "date", "content", "url"])
    writer.writerow(row)

print(" 抓取成功")
print("Title:", title)
print("Author:", author)
print("Date:", date)
print("Length:", len(content_text))


HTTPError: 403 Client Error: Forbidden for url: https://www.rfi.fr/fr/sport/20260118-un-penalty-un-boycott-une-panenka-rat%C3%A9e-les-folles-minutes-de-la-finale-s%C3%A9n%C3%A9gal-maroc

In [ ]:
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,623 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,600 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,411 kB]
Get:14 htt

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time
import csv

url = "https://www.rfi.fr/fr/sport/20260118-un-penalty-un-boycott-une-panenka-rat%C3%A9e-les-folles-minutes-de-la-finale-s%C3%A9n%C3%A9gal-maroc"

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0")

driver = webdriver.Chrome(options=options)
driver.get(url)

time.sleep(5)  # 等待 Cloudflare + JS 渲染

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, "html.parser")

# -------- title --------
title_tag = soup.select_one("h1.a-page-title")
title = title_tag.get_text(strip=True) if title_tag else ""

# -------- date --------
date_tag = soup.find("time")
date = date_tag.get_text(strip=True) if date_tag else ""

# -------- author --------
author_tag = soup.select_one("a.m-from-author__name")
author = author_tag.get_text(strip=True) if author_tag else ""

# -------- content --------
content_parts = []

chapo = soup.select_one("p.t-content__chapo")
if chapo:
    content_parts.append(chapo.get_text(" ", strip=True))

for p in soup.select("div.t-content p"):
    txt = p.get_text(" ", strip=True)
    if txt:
        content_parts.append(txt)

strong_tag = soup.select_one("div.t-content strong")
if strong_tag:
    content_parts.append(strong_tag.get_text(" ", strip=True))

content_text = "\n\n".join(content_parts)

# -------- save --------
row = [author, title, date, content_text, url]

out_file = "rfi_article.csv"
with open(out_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["author", "title", "date", "content", "url"])
    writer.writerow(row)

print(" 抓取成功")
print("Title:", title)
print("Author:", author)
print("Date:", date)
print("Content length:", len(content_text))


SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x56aea505d40a <unknown>
#1 0x56aea4a9176b <unknown>
#2 0x56aea4acb88d <unknown>
#3 0x56aea4ac7243 <unknown>
#4 0x56aea4b16dcc <unknown>
#5 0x56aea4b164ec <unknown>
#6 0x56aea4ad57b2 <unknown>
#7 0x56aea4ad6461 <unknown>
#8 0x56aea5025df9 <unknown>
#9 0x56aea5028d5d <unknown>
#10 0x56aea500eb41 <unknown>
#11 0x56aea502993b <unknown>
#12 0x56aea4ff5870 <unknown>
#13 0x56aea504a828 <unknown>
#14 0x56aea504a9fc <unknown>
#15 0x56aea505c763 <unknown>
#16 0x7d6a089a0ac3 <unknown>


#4） GDELT Project（gdeltproject.org），全球新闻数据库，包含大量法语新闻，2015–2025可覆盖。


#5）  News on the Web (NOW) Corpus，包含法语新闻，可以按年份导出。